# Debug: why 2-bit QAT (started from the 4-bit model) blows up
Not part of the submission. Each test starts from the trained 4-bit checkpoint, trains the 2-bit model for one epoch with a different setup, and logs the loss, the largest clipping-value (alpha) gradient and the range of all alphas. Run all cells top to bottom (about 3–4 minutes on a GPU).

In [ ]:
import argparse
import os
import time
import shutil

import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
import torch.backends.cudnn as cudnn

    

import torchvision
import torchvision.transforms as transforms

import sys
sys.path.insert(0, os.path.abspath('..'))   # this notebook is in software/hw/, the models/ folder is in software/
from models import *   # bring everything in the folder models

global best_prec
use_gpu = torch.cuda.is_available()
device = 'cuda' if use_gpu else 'cpu'
print('device:', device)
print('=> Building model...')
    
    
batch_size = 128

model_name = "resnet20_quant"
model = resnet20_quant()   # 4-bit QuantConv2d in every BasicBlock and both 1x1 downsample convs

        

normalize = transforms.Normalize(mean=[0.491, 0.482, 0.447], std=[0.247, 0.243, 0.262])


train_dataset = torchvision.datasets.CIFAR10(
    root='../data',
    train=True,
    download=True,
    transform=transforms.Compose([
        transforms.RandomCrop(32, padding=4),
        transforms.RandomHorizontalFlip(),
        transforms.ToTensor(),
        normalize,
    ]))
trainloader = torch.utils.data.DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=2)


test_dataset = torchvision.datasets.CIFAR10(
    root='../data',
    train=False,
    download=True,
    transform=transforms.Compose([
        transforms.ToTensor(),
        normalize,
    ]))

testloader = torch.utils.data.DataLoader(test_dataset, batch_size=batch_size, shuffle=False, num_workers=2)


print_freq = 100 # every 100 batches, accuracy printed. Here, each batch includes "batch_size" data points
# CIFAR10 has 50,000 training data, and 10,000 validation data.

def train(trainloader, model, criterion, optimizer, epoch):
    batch_time = AverageMeter()   ## at the begining of each epoch, this should be reset
    data_time = AverageMeter()
    losses = AverageMeter()
    top1 = AverageMeter()

    model.train()

    end = time.time()  # measure current time
    
    for i, (input, target) in enumerate(trainloader):
        # measure data loading time
        data_time.update(time.time() - end)  # data loading time

        input, target = input.to(device), target.to(device)

        # compute output
        output = model(input)
        loss = criterion(output, target)

        # measure accuracy and record loss
        prec = accuracy(output, target)[0]
        losses.update(loss.item(), input.size(0))
        top1.update(prec.item(), input.size(0))

        # compute gradient and do SGD step
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        # measure elapsed time
        batch_time.update(time.time() - end) # time spent to process one batch
        end = time.time()


        if i % print_freq == 0:
            print('Epoch: [{0}][{1}/{2}]\t'
                  'Time {batch_time.val:.3f} ({batch_time.avg:.3f})\t'
                  'Data {data_time.val:.3f} ({data_time.avg:.3f})\t'
                  'Loss {loss.val:.4f} ({loss.avg:.4f})\t'
                  'Prec {top1.val:.3f}% ({top1.avg:.3f}%)'.format(
                   epoch, i, len(trainloader), batch_time=batch_time,
                   data_time=data_time, loss=losses, top1=top1))

            

def validate(val_loader, model, criterion ):
    batch_time = AverageMeter()
    losses = AverageMeter()
    top1 = AverageMeter()

    # switch to evaluate mode
    model.eval()

    end = time.time()
    with torch.no_grad():
        for i, (input, target) in enumerate(val_loader):
         
            input, target = input.to(device), target.to(device)

            # compute output
            output = model(input)
            loss = criterion(output, target)

            # measure accuracy and record loss
            prec = accuracy(output, target)[0]
            losses.update(loss.item(), input.size(0))
            top1.update(prec.item(), input.size(0))

            # measure elapsed time
            batch_time.update(time.time() - end)
            end = time.time()

            if i % print_freq == 0:  # This line shows how frequently print out the status. e.g., i%5 => every 5 batch, prints out
                print('Test: [{0}/{1}]\t'
                  'Time {batch_time.val:.3f} ({batch_time.avg:.3f})\t'
                  'Loss {loss.val:.4f} ({loss.avg:.4f})\t'
                  'Prec {top1.val:.3f}% ({top1.avg:.3f}%)'.format(
                   i, len(val_loader), batch_time=batch_time, loss=losses,
                   top1=top1))

    print(' * Prec {top1.avg:.3f}% '.format(top1=top1))
    return top1.avg


def accuracy(output, target, topk=(1,5)):
    """Computes the precision@k for the specified values of k"""
    maxk = max(topk) # 5
    batch_size = target.size(0) # 128

    _, pred = output.topk(maxk, 1, True, True) # topk(k, dim=None, largest=True, sorted=True)
                                    # will output (max value, its index)
    pred = pred.t()               # transpose
    correct = pred.eq(target.view(1, -1).expand_as(pred))   # "-1": calculate automatically

    res = []
    for k in topk: # 1, 5
        correct_k = correct[:k].reshape(-1).float().sum(0)  # view(-1): make a flattened 1D tensor
        res.append(correct_k.mul_(100.0 / batch_size))   # correct: size of [maxk, batch_size]
    return res


class AverageMeter(object):
    """Computes and stores the average and current value"""
    def __init__(self):
        self.reset()

    def reset(self):
        self.val = 0
        self.avg = 0
        self.sum = 0
        self.count = 0

    def update(self, val, n=1):
        self.val = val
        self.sum += val * n    ## n is impact factor
        self.count += n
        self.avg = self.sum / self.count

        
def save_checkpoint(state, is_best, fdir):
    filepath = os.path.join(fdir, 'checkpoint.pth')
    torch.save(state, filepath)
    if is_best:
        shutil.copyfile(filepath, os.path.join(fdir, 'model_best.pth.tar'))


def adjust_learning_rate(optimizer, epoch):
    """For resnet, the lr starts from 0.1, and is divided by 10 at 80 and 120 epochs"""
    adjust_list = [150, 225]
    if epoch in adjust_list:
        for param_group in optimizer.param_groups:
            param_group['lr'] = param_group['lr'] * 0.1        

#model = nn.DataParallel(model).cuda()
#all_params = checkpoint['state_dict']
#model.load_state_dict(all_params, strict=False)
#criterion = nn.CrossEntropyLoss().cuda()
#validate(testloader, model, criterion)

In [ ]:
from models.resnet_quant_2bit import resnet20_quant_2bit

STEPS = 391          # one epoch per test
criterion = nn.CrossEntropyLoss().to(device)
ckpt_4bit = torch.load('result/resnet20_quant/model_best.pth.tar', map_location=device)['state_dict']
images_dbg, labels_dbg = next(iter(trainloader))

def make_2bit(from_4bit=True, wgt_alpha=None, act_alpha=None):
    # wgt_alpha / act_alpha: reset every 2-bit clipping value to this start value (None = keep).
    # The first conv's act_alpha always stays 2.0 (image range).
    m = resnet20_quant_2bit().to(device)
    if from_4bit:
        m.load_state_dict(ckpt_4bit, strict=False)
    with torch.no_grad():
        for n, l in m.named_modules():
            if hasattr(l, 'act_alpha'):
                if wgt_alpha is not None:
                    l.weight_quant.wgt_alpha.fill_(wgt_alpha)
                if act_alpha is not None and n != 'conv1':
                    l.act_alpha.fill_(act_alpha)
    return m

def alpha_summary(m):
    act = [l.act_alpha.item() for l in m.modules() if hasattr(l, 'act_alpha')]
    wgt = [l.weight_quant.wgt_alpha.item() for l in m.modules() if hasattr(l, 'weight_quant')]
    return 'act_alpha [{:9.2f}, {:9.2f}]  wgt_alpha [{:8.2f}, {:8.2f}]'.format(min(act), max(act), min(wgt), max(wgt))

def dead_channels(m):
    # all-zero 2-bit filters, and BatchNorm channels whose input has zero variance (these blow up the gradient)
    zero_var = []
    hs = [l.register_forward_hook(lambda mod, inp, out: zero_var.append((inp[0].var(dim=(0, 2, 3)) < 1e-8).sum().item()))
          for l in m.modules() if isinstance(l, nn.BatchNorm2d)]
    m.train()
    with torch.no_grad():
        m(images_dbg.to(device))
    for h in hs:
        h.remove()
    zero_filters = sum((l.weight_q.detach().abs().sum(dim=(1, 2, 3)) == 0).sum().item()
                       for l in m.modules() if hasattr(l, 'act_alpha'))
    return 'all-zero filters: {}, zero-variance BN channels: {}'.format(zero_filters, sum(zero_var))

def test_run(name, lr=0.05, clip_norm=None, every=40, **make_kw):
    torch.manual_seed(0)
    m = make_2bit(**make_kw)
    alpha_params = [p for n, p in m.named_parameters() if 'alpha' in n]
    other_params = [p for n, p in m.named_parameters() if 'alpha' not in n and 'weight_q' not in n]
    opt = torch.optim.SGD([{'params': other_params, 'weight_decay': 5e-4},
                           {'params': alpha_params, 'weight_decay': 0}], lr=lr, momentum=0.9)
    print('=====', name)
    print('at start:', dead_channels(m), '|', alpha_summary(m))
    m.train()
    for step, (input, target) in enumerate(trainloader):
        if step >= STEPS:
            break
        input, target = input.to(device), target.to(device)
        loss = criterion(m(input), target)
        opt.zero_grad()
        loss.backward()
        g = max(p.grad.abs().max().item() for p in other_params + alpha_params if p.grad is not None)
        if clip_norm is not None:
            torch.nn.utils.clip_grad_norm_(other_params + alpha_params, clip_norm)
        opt.step()
        if step < 5 or step % every == 0:
            print('step {:3d}  loss {:9.4f}  max|grad| {:10.3g}  {}'.format(step, loss.item(), g, alpha_summary(m)))
    acc = validate(testloader, m, criterion)
    print('>>> {}: test acc after 1 epoch = {:.2f}%   {}\n'.format(name, acc, alpha_summary(m)))
    return acc

In [ ]:
results = {}
results['A: 4-bit start, 4-bit alphas (original)']    = test_run('A: 4-bit start, 4-bit alphas (original)')
results['G: 4-bit start, wgt 1.0 / act 3.0']          = test_run('G: 4-bit start, wgt 1.0 / act 3.0', wgt_alpha=1.0, act_alpha=3.0)
results['H: 4-bit start, wgt 1.5 / act 3.0']          = test_run('H: 4-bit start, wgt 1.5 / act 3.0', wgt_alpha=1.5, act_alpha=3.0)
results['I: 4-bit start, wgt 1.0 / act 2.0']          = test_run('I: 4-bit start, wgt 1.0 / act 2.0', wgt_alpha=1.0, act_alpha=2.0)
results['J: G + grad-norm clip 5']                    = test_run('J: G + grad-norm clip 5', wgt_alpha=1.0, act_alpha=3.0, clip_norm=5)
results['K: scratch, default alphas (3.0 / 8.0)']     = test_run('K: scratch, default alphas (3.0 / 8.0)', from_4bit=False, lr=0.1)
results['L: scratch, wgt 1.0 / act 3.0']              = test_run('L: scratch, wgt 1.0 / act 3.0', from_4bit=False, lr=0.1, wgt_alpha=1.0, act_alpha=3.0)

In [ ]:
for name, acc in results.items():
    print('{:45s} {:6.2f}%'.format(name, acc))